# Upgraded Hybrid Data Merger — Enrichment
This notebook loads `master_student_data.csv` and **infers missing ML features** (latest_cgpa, semesters_count, trends, volatilities, normalized external marks, etc.).

In [8]:
# CELL 1 - Imports & config
import os, json, ast, logging, re
import numpy as np, pandas as pd
from typing import Any, Dict
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
MASTER_IN = 'master_student_data.csv'
MASTER_OUT = 'master_student_data_enriched.csv'
print('Input:', MASTER_IN, 'Output:', MASTER_OUT)

Input: master_student_data.csv Output: master_student_data_enriched.csv


In [9]:
# CELL 2 - Load master CSV
if not os.path.exists(MASTER_IN):
    raise FileNotFoundError(MASTER_IN + ' not found in notebook working directory.')
df = pd.read_csv(MASTER_IN)
print('Loaded rows,cols:', df.shape)
print('Columns:', df.columns.tolist())

Loaded rows,cols: (211, 34)
Columns: ['_id', 'userid', '__v', 'createdat', 'localentry', 'puc', 'sslc', 'updatedat', 'localentry_parsed', 'puc_parsed', 'sslc_parsed', 'sslc_percentage', 'sslc_board', 'sslc_year', 'puc_percentage', 'puc_board', 'puc_year', 'overall_attendance', 'latest_semester_attendance', 'attendance_trend', 'subjects_attended', 'avg_iat_score', 'iat_subjects_count', 'iat_performance', 'avg_external_marks', 'avg_cgpa', 'pass_rate', 'external_subjects_count', 'internship_count', 'has_internship', 'activity_count', 'has_activities', 'tyl_skills_tracked', 'tyl_latest_semester']


In [10]:
# CELL 3 - Utility helpers
import math
def to_number(x, default=0.0):
    try:
        if pd.isna(x): return default
        if isinstance(x, (int,float,np.number)): return float(x)
        s = str(x).strip()
        if s=='' or s.lower() in {'nan','none','null'}: return default
        # extract first number
        m = re.search(r"\d+\.?\d*", s)
        if m: return float(m.group())
        return default
    except:
        return default

def safe_eval_json_like(val):
    if pd.isna(val) or val=='' or val=='[]' or val=='{}' or val=='nan': return None
    if isinstance(val, (dict,list)): return val
    try:
        # replace ObjectId('...') and datetime.datetime(...) patterns
        v = re.sub(r"ObjectId\(['\"]([^'\"]+)['\"]\)", r'"\\1"', str(val))
        v = re.sub(r'datetime\.datetime\([^\)]*\)', 'null', v)
        return ast.literal_eval(v)
    except Exception:
        # try JSON
        try:
            return json.loads(val)
        except Exception:
            return None

In [11]:
# CELL 4 - Normalize and infer fields
df2 = df.copy()
# Normalize avg_external_marks: detect if mostly <=50 then scale to 100
if 'avg_external_marks' in df2.columns:
    vals = pd.to_numeric(df2['avg_external_marks'], errors='coerce')
    median = vals.median(skipna=True)
    if not np.isnan(median) and median <= 55:
        logging.info('Detected external marks likely out of 50 scale; scaling by *2 to 0-100.')
        df2['avg_external_marks_normalized'] = vals.fillna(0) * 2.0
    else:
        df2['avg_external_marks_normalized'] = vals.fillna(0)
else:
    df2['avg_external_marks_normalized'] = 0.0

# Ensure numeric columns exist
for col in ['avg_cgpa','avg_iat_score','overall_attendance','pass_rate','internship_count','activity_count','tyl_skills_tracked']:
    if col not in df2.columns:
        df2[col] = 0

# Infer semesters_count from parsed fields
def infer_semesters_count_row(row):
    for candidate in ['localentry_parsed','puc_parsed','sslc_parsed','semesters_parsed']:
        if candidate in row and not pd.isna(row[candidate]):
            parsed = safe_eval_json_like(row[candidate])
            if isinstance(parsed, list):
                return len(parsed)
            if isinstance(parsed, dict):
                # dict mapping semester->info
                return len(parsed.keys())
    # fallback: if iat_subjects_count exists, assume at least 1
    if 'iat_subjects_count' in row and not pd.isna(row['iat_subjects_count']):
        try:
            v = int(to_number(row['iat_subjects_count'],0))
            return max(1, v//1)
        except:
            return 1
    return 1

df2['semesters_count'] = df2.apply(infer_semesters_count_row, axis=1)

# Infer latest_cgpa: prefer column avg_cgpa if latest missing
if 'latest_cgpa' not in df2.columns:
    df2['latest_cgpa'] = pd.to_numeric(df2.get('avg_cgpa', 0), errors='coerce').fillna(0)

# first_cgpa fallback
if 'first_cgpa' not in df2.columns:
    df2['first_cgpa'] = df2['latest_cgpa']

2025-11-13 21:18:16,505 INFO: Detected external marks likely out of 50 scale; scaling by *2 to 0-100.


In [12]:
# CELL 5 - Compute simple trend/volatility placeholders (safe defaults)
# Because we lack per-semester values, set sensible defaults.
df2['cgpa_trend'] = 'stable'    # default
df2['cgpa_volatility'] = 0.0
df2['cgpa_improvement_rate'] = 0.0

df2['iat_trend'] = 'stable'
df2['iat_volatility'] = 0.0
df2['iat_improvement_rate'] = 0.0

df2['attendance_trend'] = 'stable'
df2['attendance_volatility'] = 0.0
df2['attendance_change'] = 0.0

# If there is a parsed semesters field with per-semester CGPA, compute real values
def extract_cgpa_from_semesters(parsed_obj):
    try:
        if not parsed_obj: return []
        if isinstance(parsed_obj, list):
            vals = []
            for sem in parsed_obj:
                if isinstance(sem, dict):
                    for subj in sem.get('subjects', []) if 'subjects' in sem else []:
                        if 'cgpa' in subj:
                            vals.append(to_number(subj['cgpa']))
            return vals
        if isinstance(parsed_obj, dict):
            vals = []
            for k,v in parsed_obj.items():
                if isinstance(v, dict) and 'subjects' in v:
                    for subj in v.get('subjects',[]):
                        if 'cgpa' in subj:
                            vals.append(to_number(subj['cgpa']))
            return vals
    except Exception:
        return []

# Try to populate cgpa trajectory if possible
cgpa_list_all = []
for idx,row in df2.iterrows():
    parsed = None
    if 'externals_parsed' in row and not pd.isna(row['externals_parsed']):
        parsed = safe_eval_json_like(row['externals_parsed'])
    elif 'semesters_parsed' in row and not pd.isna(row['semesters_parsed']):
        parsed = safe_eval_json_like(row['semesters_parsed'])
    vals = extract_cgpa_from_semesters(parsed)
    if vals and len(vals) > 0:
        first = vals[0]; latest = vals[-1]
        df2.at[idx,'first_cgpa'] = first
        df2.at[idx,'latest_cgpa'] = latest
        if len(vals) >= 2:
            df2.at[idx,'cgpa_improvement_rate'] = (latest - first)/len(vals)
            df2.at[idx,'cgpa_volatility'] = float(np.std(vals))
            df2.at[idx,'cgpa_trend'] = 'improving' if latest > first else ('declining' if latest < first else 'stable')

In [13]:
# CELL 6 - Derive engagement/experience features
# internships/activity diversity approximations
df2['internship_diversity_score'] = 0.0
df2['activity_diversity_score'] = 0.0
if 'internship_count' in df2.columns:
    df2['internship_diversity_score'] = df2['internship_count'].apply(lambda x: min(100, to_number(x)*25))
if 'activity_count' in df2.columns:
    df2['activity_diversity_score'] = df2['activity_count'].apply(lambda x: min(100, to_number(x)*20))

# engagement score approximate
df2['engagement_score'] = ((pd.to_numeric(df2.get('internship_count',0),errors='coerce').fillna(0)*10) +
                           (pd.to_numeric(df2.get('activity_count',0),errors='coerce').fillna(0)*2) +
                           (pd.to_numeric(df2.get('tyl_skills_tracked',0),errors='coerce').fillna(0)*5)).clip(0,100)

In [ ]:
# CELL X — Safe composite feature calculation

def safe_col(df, col, default=0):
    """Return a safe Series (correct length), not a scalar."""
    if col in df.columns:
        return pd.to_numeric(df[col], errors='coerce').fillna(default)
    else:
        return pd.Series([default] * len(df))

def compute_composites(df):

    # --- Weighted Foundation ---
    sslc = safe_col(df, 'sslc_percentage', 0)
    puc = safe_col(df, 'puc_percentage', 0)
    local = safe_col(df, 'local_percentage', 0)

    df['weighted_foundation'] = (
        sslc * 0.3 +
        puc * 0.5 +
        local * 0.2
    )

    # --- Academic Health Index ---
    avg_cgpa = safe_col(df, 'avg_cgpa', 0)
    avg_iat = safe_col(df, 'avg_iat_score', 0)
    att = safe_col(df, 'overall_attendance', 0)

    df['academic_health_index'] = (
        avg_cgpa * 10 * 0.5 +
        avg_iat * 0.3 +
        att * 0.2
    )

    # --- Profile Diversity ---
    intern_div = safe_col(df, 'internship_diversity_score', 0)
    act_div = safe_col(df, 'activity_diversity_score', 0)

    df['profile_diversity'] = (
        intern_div * 0.4 +
        act_div * 0.6
    )

    # --- Consistency Index (reverse volatility) ---
    cgpa_vol = safe_col(df, 'cgpa_volatility', 0)
    iat_vol = safe_col(df, 'iat_volatility', 0)
    att_vol = safe_col(df, 'attendance_volatility', 0)

    df['overall_volatility'] = cgpa_vol + iat_vol + att_vol
    df['consistency_index'] = 100 - df['overall_volatility']

    return df


In [15]:
# CELL 8 - Final cleanups and save
# Fill NaNs and types
for col in df2.columns:
    if df2[col].dtype == object:
        # try numeric conversion
        try:
            df2[col] = pd.to_numeric(df2[col], errors='coerce')
        except:
            pass
# Fill numeric NaNs
numcols = df2.select_dtypes(include=[np.number]).columns
df2[numcols] = df2[numcols].fillna(0)

# Ensure booleans
for b in ['has_internship','has_activities']:
    if b in df2.columns:
        df2[b] = df2[b].fillna(False)

# Save
df2.to_csv(MASTER_OUT, index=False)
print('Saved enriched master to', MASTER_OUT)
df2.head(3)

Saved enriched master to master_student_data_enriched.csv


,_id,userid,__v,createdat,localentry,puc,sslc,updatedat,localentry_parsed,puc_parsed,...,cgpa_volatility,cgpa_improvement_rate,iat_trend,iat_volatility,iat_improvement_rate,attendance_volatility,attendance_change,internship_diversity_score,activity_diversity_score,engagement_score
0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
